In [ ]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class GoldTradingSystem:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://www.alphavantage.co/query"
        
    def fetch_1min_data_for_month(self, symbol="GLD", month="2024-09"):
        """Fetch 1-minute data for a specific month"""
        params = {
            'function': 'TIME_SERIES_INTRADAY',
            'symbol': symbol,
            'interval': '1min',
            'month': month,
            'outputsize': 'full',
            'apikey': self.api_key
        }
        
        try:
            response = requests.get(self.base_url, params=params)
            data = response.json()
            
            if 'Time Series (1min)' in data:
                time_series = data['Time Series (1min)']
                df = pd.DataFrame.from_dict(time_series, orient='index')
                df.reset_index(inplace=True)
                df.rename(columns={
                    'index': 'datetime',
                    '1. open': 'open', '2. high': 'high',
                    '3. low': 'low', '4. close': 'close', '5. volume': 'volume'
                }, inplace=True)
                
                df['datetime'] = pd.to_datetime(df['datetime'])
                df = df.sort_values('datetime').reset_index(drop=True)
                
                for col in ['open', 'high', 'low', 'close', 'volume']:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                
                return df
            else:
                return None
        except Exception as e:
            print(f"Error: {e}")
            return None

    def get_last_30_days_complete(self):
        """Get complete 30 days by fetching multiple months"""
        end_date = datetime.now()
        start_date = end_date - timedelta(days=30)
        
        # Calculate months needed
        months_needed = []
        current_check = start_date
        while current_check <= end_date:
            month_str = current_check.strftime("%Y-%m")
            if month_str not in months_needed:
                months_needed.append(month_str)
            if current_check.month == 12:
                current_check = current_check.replace(year=current_check.year + 1, month=1)
            else:
                current_check = current_check.replace(month=current_check.month + 1)
        
        # Try different Gold symbols
        symbols_to_try = ['GLD', 'GOLD', 'IAU', 'XAUUSD']
        all_data = []
        
        for symbol in symbols_to_try:
            symbol_data = []
            for month in months_needed:
                month_data = self.fetch_1min_data_for_month(symbol, month)
                if month_data is not None:
                    symbol_data.append(month_data)
            
            if symbol_data:
                all_data = symbol_data
                break
        
        if all_data:
            combined_df = pd.concat(all_data, ignore_index=True)
            combined_df = combined_df.drop_duplicates(subset=['datetime']).sort_values('datetime').reset_index(drop=True)
            cutoff_date = end_date - timedelta(days=30)
            return combined_df[combined_df['datetime'] >= cutoff_date].copy()
        
        return None

    def save_data_to_csv(self, data, filename):
        """Save data to CSV file"""
        try:
            data.to_csv(filename, index=False)
            print(f"💾 Saved: {filename} ({len(data):,} rows)")
            return filename
        except Exception as e:
            print(f"❌ Error saving: {e}")
            return None

API_KEY = "YOUR_ALPHA_VANTAGE_API_KEY"  
gold_system = GoldTradingSystem(API_KEY)

print("📊 Fetching 1-minute Gold data for last 30 days...")
data_1min = gold_system.get_last_30_days_complete()

if data_1min is not None:
    print(f"✅ Got {len(data_1min):,} 1-minute candles")
    gold_system.save_data_to_csv(data_1min, "gold_1min_raw_data.csv")
    print(data_1min.tail())
else:
    print("❌ No data fetched")

In [3]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv("/Users/mac/Desktop/ForxTrading/experiments/gold_1min_complete_30days.csv")
df.shape

(15702, 7)

In [7]:
def calculate_traditional_indicators(df):
    """Add traditional technical indicators"""
    
    # Simple Moving Averages
    df['SMA_20'] = df['close'].rolling(20).mean()
    df['SMA_50'] = df['close'].rolling(50).mean()
    df['SMA_200'] = df['close'].rolling(200).mean()
    
    # Exponential Moving Averages  
    df['EMA_10'] = df['close'].ewm(span=10).mean()
    df['EMA_21'] = df['close'].ewm(span=21).mean()
    
    # RSI (Relative Strength Index)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    
    # MACD
    exp1 = df['close'].ewm(span=12).mean()
    exp2 = df['close'].ewm(span=26).mean()
    df['MACD'] = exp1 - exp2
    df['MACD_Signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_Histogram'] = df['MACD'] - df['MACD_Signal']
    
    # Bollinger Bands
    sma20 = df['close'].rolling(20).mean()
    std20 = df['close'].rolling(20).std()
    df['BB_Upper'] = sma20 + (std20 * 2)
    df['BB_Lower'] = sma20 - (std20 * 2)
    df['BB_Middle'] = sma20
    
    # Stochastic Oscillator
    low_min = df['low'].rolling(14).min()
    high_max = df['high'].rolling(14).max()
    df['Stoch_K'] = 100 * ((df['close'] - low_min) / (high_max - low_min))
    df['Stoch_D'] = df['Stoch_K'].rolling(3).mean()
    
    # ATR (Average True Range)
    df['prev_close'] = df['close'].shift(1)
    df['tr1'] = df['high'] - df['low']
    df['tr2'] = abs(df['high'] - df['prev_close'])
    df['tr3'] = abs(df['low'] - df['prev_close'])
    df['true_range'] = df[['tr1', 'tr2', 'tr3']].max(axis=1)
    df['ATR'] = df['true_range'].rolling(14).mean()
    df.drop(['prev_close', 'tr1', 'tr2', 'tr3'], axis=1, inplace=True)
    
    return df

# Add traditional indicators
print("📈 Adding Traditional Technical Indicators...")
df = calculate_traditional_indicators(df.copy())
print(f"✅ Added traditional indicators. Dataset now has {len(df.columns)} columns")

📈 Adding Traditional Technical Indicators...
✅ Added traditional indicators. Dataset now has 23 columns


In [8]:
def calculate_smc_indicators(df):
    """Add Smart Money Concepts indicators"""
    
    # Break of Structure (BOS) - Market structure changes
    df['SMC_BOS_bullish'] = (df['high'] > df['high'].rolling(10).max().shift(1)).astype(int)
    df['SMC_BOS_bearish'] = (df['low'] < df['low'].rolling(10).min().shift(1)).astype(int)
    
    # Order Blocks - Institutional supply/demand zones
    bullish_move = df['close'] > df['close'].shift(5)
    bearish_candle = df['close'] < df['open']
    df['SMC_bullish_ob'] = (bearish_candle.shift(1) & bullish_move).astype(int)
    
    bearish_move = df['close'] < df['close'].shift(5)
    bullish_candle = df['close'] > df['open']
    df['SMC_bearish_ob'] = (bullish_candle.shift(1) & bearish_move).astype(int)
    
    # Fair Value Gaps (FVG) - Price imbalances
    df['SMC_bullish_fvg'] = ((df['low'] > df['high'].shift(2)) & 
                             (df['close'] > df['close'].shift(1))).astype(int)
    df['SMC_bearish_fvg'] = ((df['high'] < df['low'].shift(2)) & 
                             (df['close'] < df['close'].shift(1))).astype(int)
    
    # Market Structure - Overall trend direction
    hh_hl = (df['high'] > df['high'].shift(10)) & (df['low'] > df['low'].shift(10))
    ll_lh = (df['low'] < df['low'].shift(10)) & (df['high'] < df['high'].shift(10))
    df['SMC_market_structure'] = np.where(hh_hl, 1, np.where(ll_lh, -1, 0))
    
    # Liquidity Sweeps - Stop hunting by institutions
    df['SMC_liquidity_sweep_high'] = (df['high'] > df['high'].rolling(20).max().shift(1)).astype(int)
    df['SMC_liquidity_sweep_low'] = (df['low'] < df['low'].rolling(20).min().shift(1)).astype(int)
    
    return df


print("🧠 Adding Smart Money Concepts (SMC)...")
df = calculate_smc_indicators(df)
smc_count = len([col for col in df.columns if col.startswith('SMC_')])
print(f"✅ Added {smc_count} SMC indicators")

🧠 Adding Smart Money Concepts (SMC)...
✅ Added 9 SMC indicators


In [9]:
def calculate_wyckoff_indicators(df):
    """Add Wyckoff Method indicators"""
    
    # Volume analysis
    vol_ma_20 = df['volume'].rolling(20).mean()
    df['Wyckoff_volume_ratio'] = df['volume'] / vol_ma_20
    
    # Price Spread Analysis
    price_spread = df['high'] - df['low']
    spread_ma = price_spread.rolling(20).mean()
    
    # Effort vs Result - Volume vs Price movement relationship
    df['Wyckoff_effort_result'] = (df['volume'] / vol_ma_20) / (price_spread / spread_ma)
    
    # Volume anomalies
    df['Wyckoff_volume_spike'] = (df['volume'] > vol_ma_20 * 2).astype(int)
    df['Wyckoff_high_vol_narrow_spread'] = ((df['volume'] > vol_ma_20 * 1.5) & 
                                            (price_spread < spread_ma)).astype(int)
    
    # Accumulation/Distribution phases
    price_range_std = df['close'].rolling(30).std()
    is_range_bound = price_range_std < price_range_std.rolling(100).mean()
    
    up_movement = df['close'] > df['close'].shift(1)
    down_movement = df['close'] < df['close'].shift(1)
    
    df['Wyckoff_accumulation'] = (is_range_bound & up_movement & 
                                  (df['volume'] > vol_ma_20)).astype(int)
    df['Wyckoff_distribution'] = (is_range_bound & down_movement & 
                                  (df['volume'] > vol_ma_20)).astype(int)
    
    # Volume Price Trend
    df['Wyckoff_vpt'] = (df['volume'] * df['close'].pct_change()).cumsum()
    df['Wyckoff_vpt_ma'] = df['Wyckoff_vpt'].rolling(50).mean()
    df['Wyckoff_vpt_signal'] = np.where(df['Wyckoff_vpt'] > df['Wyckoff_vpt_ma'], 1, -1)
    
    # Test phases and climactic action
    df['Wyckoff_test_phase'] = (df['volume'] < vol_ma_20 * 0.7).astype(int)
    df['Wyckoff_climax'] = (df['volume'] > vol_ma_20 * 3).astype(int)
    
    return df

# Add Wyckoff indicators
print("📊 Adding Wyckoff Method indicators...")
df = calculate_wyckoff_indicators(df)
wyckoff_count = len([col for col in df.columns if col.startswith('Wyckoff_')])
print(f"✅ Added {wyckoff_count} Wyckoff indicators")

📊 Adding Wyckoff Method indicators...
✅ Added 11 Wyckoff indicators


In [10]:
def calculate_ict_indicators(df):
    """Add ICT concepts"""
    
    # Market Maker Buy/Sell Model - Liquidity raids + reversals
    high_raid = df['high'] > df['high'].rolling(15).max().shift(1)
    low_raid = df['low'] < df['low'].rolling(15).min().shift(1)
    
    df['ICT_mmbm_buy_setup'] = (low_raid.shift(1) & (df['close'] > df['open'])).astype(int)
    df['ICT_mmbm_sell_setup'] = (high_raid.shift(1) & (df['close'] < df['open'])).astype(int)
    
    # Optimal Trade Entry (OTE) - Fibonacci retracement zones
    swing_high = df['high'].rolling(20).max()
    swing_low = df['low'].rolling(20).min()
    
    fib_618 = swing_high - (swing_high - swing_low) * 0.618
    fib_786 = swing_high - (swing_high - swing_low) * 0.786
    df['ICT_ote_buy_zone'] = ((df['low'] <= fib_786) & (df['high'] >= fib_618)).astype(int)
    
    fib_buy_618 = swing_low + (swing_high - swing_low) * 0.618
    fib_buy_786 = swing_low + (swing_high - swing_low) * 0.786
    df['ICT_ote_sell_zone'] = ((df['high'] >= fib_buy_618) & (df['low'] <= fib_buy_786)).astype(int)
    
    # Displacement - Strong institutional moves
    displacement_up = (df['close'] > df['close'].shift(5)) & (df['low'] > df['low'].shift(5))
    displacement_down = (df['close'] < df['close'].shift(5)) & (df['high'] < df['high'].shift(5))
    df['ICT_displacement'] = np.where(displacement_up, 1, np.where(displacement_down, -1, 0))
    
    # Order Block Mitigation
    df['ICT_ob_mitigation_bull'] = ((df['low'] <= df['low'].shift(10)) & 
                                    (df['close'] > df['open'])).astype(int)
    df['ICT_ob_mitigation_bear'] = ((df['high'] >= df['high'].shift(10)) & 
                                    (df['close'] < df['open'])).astype(int)
    
    # Market Structure Shift
    df['ICT_mss_bullish'] = (df['high'] > df['high'].rolling(10).max().shift(1)).astype(int)
    df['ICT_mss_bearish'] = (df['low'] < df['low'].rolling(10).min().shift(1)).astype(int)
    
    # Fair Value Gaps (ICT style)
    df['ICT_fvg_bullish'] = df['SMC_bullish_fvg']
    df['ICT_fvg_bearish'] = df['SMC_bearish_fvg']
    
    # Liquidity Voids and Institutional Candles
    price_gap = abs(df['open'] - df['close'].shift(1))
    avg_gap = price_gap.rolling(50).mean()
    df['ICT_liquidity_void'] = (price_gap > avg_gap * 2).astype(int)
    
    candle_body = abs(df['close'] - df['open'])
    avg_body = candle_body.rolling(50).mean()
    df['ICT_institutional_candle'] = (candle_body > avg_body * 2).astype(int)
    
    return df

# Add ICT indicators
print("⚡ Adding ICT (Inner Circle Trader) concepts...")
df = calculate_ict_indicators(df)
ict_count = len([col for col in df.columns if col.startswith('ICT_')])
print(f"✅ Added {ict_count} ICT indicators")

⚡ Adding ICT (Inner Circle Trader) concepts...
✅ Added 13 ICT indicators


In [12]:
def create_ml_labels(df, price_threshold_pct=0.1, time_horizons=[5, 15, 30, 60]):
    """Create future price movement labels for ML"""
    
    for horizon in time_horizons:
        print(f"   Creating {horizon}-minute labels...")
        
        # Calculate future returns
        future_price = df['close'].shift(-horizon)
        future_return = (future_price / df['close'] - 1) * 100
        
        # Create labels: 1=BUY, -1=SELL, 0=HOLD
        buy_condition = future_return > price_threshold_pct
        sell_condition = future_return < -price_threshold_pct
        
        df[f'future_return_{horizon}min'] = future_return
        df[f'label_{horizon}min'] = np.where(buy_condition, 1, 
                                           np.where(sell_condition, -1, 0))
        
        # Binary labels (simpler): 1=UP, 0=DOWN
        df[f'binary_label_{horizon}min'] = np.where(future_return > 0, 1, 0)
        
        # Price targets
        df[f'price_target_{horizon}min'] = future_price
        
        # Signal strength based on magnitude
        abs_return = abs(future_return)
        df[f'signal_strength_{horizon}min'] = np.where(
            abs_return > price_threshold_pct * 3, 3,
            np.where(abs_return > price_threshold_pct * 2, 2,
                    np.where(abs_return > price_threshold_pct, 1, 0))
        )
    
    return df

# Add additional features for ML
def add_ml_features(df):
    """Add extra features for machine learning"""
    
    # Price-based features
    df['price_change'] = df['close'].pct_change()
    df['price_change_5min'] = df['close'].pct_change(5)
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    
    # Volatility measures
    df['volatility_10min'] = df['close'].rolling(10).std()
    df['volatility_30min'] = df['close'].rolling(30).std()
    
    # Volume features
    df['volume_change'] = df['volume'].pct_change()
    df['volume_ma_ratio'] = df['volume'] / df['volume'].rolling(50).mean()
    
        # Time-based features (important for 1-minute data)
    df['hour'] = pd.to_datetime(df['datetime']).dt.hour
    df['minute'] = pd.to_datetime(df['datetime']).dt.minute
    df['weekday'] = pd.to_datetime(df['datetime']).dt.weekday
    df['is_market_open'] = ((df['hour'] >= 9) & (df['hour'] < 17)).astype(int)
    
    # Price position relative to bands
    df['price_position_bb'] = (df['close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])
    
    # Momentum features
    df['momentum_5min'] = df['close'] / df['close'].shift(5) - 1
    df['momentum_15min'] = df['close'] / df['close'].shift(15) - 1
    
    return df

# Create ML labels and features
print("🎯 Creating ML labels and additional features...")
df = create_ml_labels(df)
df = add_ml_features(df)

# Show label distribution
label_counts = df['label_15min'].value_counts().sort_index()
total = len(df)
print(f"\n📊 ML Labels (15-min prediction):")
print(f"   SELL (-1): {label_counts.get(-1, 0):,} ({label_counts.get(-1, 0)/total*100:.1f}%)")
print(f"   HOLD (0):  {label_counts.get(0, 0):,} ({label_counts.get(0, 0)/total*100:.1f}%)")
print(f"   BUY (1):   {label_counts.get(1, 0):,} ({label_counts.get(1, 0)/total*100:.1f}%)")

print(f"✅ Dataset now has {len(df.columns)} total features")

🎯 Creating ML labels and additional features...
   Creating 5-minute labels...
   Creating 15-minute labels...
   Creating 30-minute labels...
   Creating 60-minute labels...

📊 ML Labels (15-min prediction):
   SELL (-1): 1,370 (8.7%)
   HOLD (0):  12,846 (81.8%)
   BUY (1):   1,486 (9.5%)
✅ Dataset now has 91 total features


In [13]:
df.info(max_cols=200)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15702 entries, 0 to 15701
Data columns (total 91 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   datetime                        15702 non-null  object 
 1   open                            15702 non-null  float64
 2   high                            15702 non-null  float64
 3   low                             15702 non-null  float64
 4   close                           15702 non-null  float64
 5   volume                          15702 non-null  int64  
 6   date                            15702 non-null  object 
 7   SMA_20                          15683 non-null  float64
 8   SMA_50                          15653 non-null  float64
 9   SMA_200                         15503 non-null  float64
 10  EMA_10                          15702 non-null  float64
 11  EMA_21                          15702 non-null  float64
 12  RSI_14                          

In [14]:
df.head()

,datetime,open,high,low,close,volume,date,SMA_20,SMA_50,SMA_200,...,volatility_30min,volume_change,volume_ma_ratio,hour,minute,weekday,is_market_open,price_position_bb,momentum_5min,momentum_15min
0,2025-08-05 12:54:00,311.3400,311.3400,311.31,311.3293,3609,2025-08-05,NaN,NaN,NaN,...,NaN,NaN,NaN,12,54,1,1,NaN,NaN,NaN
1,2025-08-05 12:55:00,311.3383,311.3900,311.33,311.3400,4762,2025-08-05,NaN,NaN,NaN,...,NaN,0.319479,NaN,12,55,1,1,NaN,NaN,NaN
2,2025-08-05 12:56:00,311.3400,311.4100,311.34,311.4100,3270,2025-08-05,NaN,NaN,NaN,...,NaN,-0.313314,NaN,12,56,1,1,NaN,NaN,NaN
3,2025-08-05 12:57:00,311.3900,311.4100,311.38,311.4100,2183,2025-08-05,NaN,NaN,NaN,...,NaN,-0.332416,NaN,12,57,1,1,NaN,NaN,NaN
4,2025-08-05 12:58:00,311.4200,311.4399,311.38,311.3950,8296,2025-08-05,NaN,NaN,NaN,...,NaN,2.800275,NaN,12,58,1,1,NaN,NaN,NaN


In [15]:
df.tail()

,datetime,open,high,low,close,volume,date,SMA_20,SMA_50,SMA_200,...,volatility_30min,volume_change,volume_ma_ratio,hour,minute,weekday,is_market_open,price_position_bb,momentum_5min,momentum_15min
15697,2025-08-29 19:56:00,318.0700,318.0700,318.00,318.00,3,2025-08-29,317.997555,318.024894,317.793548,...,0.060046,-0.918919,0.023102,19,56,4,0,0.509348,0.000315,0.000063
15698,2025-08-29 19:57:00,318.0973,318.0973,318.05,318.05,205,2025-08-29,318.000055,318.024294,317.796348,...,0.058842,67.333333,1.592604,19,57,4,0,0.687956,0.000157,0.000126
15699,2025-08-29 19:58:00,318.0500,318.1000,318.05,318.10,48,2025-08-29,318.005555,318.025294,317.799348,...,0.061018,-0.765854,0.390752,19,58,4,0,0.837243,0.000313,0.000094
15700,2025-08-29 19:59:00,318.1000,318.1000,318.06,318.10,1001,2025-08-29,318.008055,318.026294,317.802348,...,0.063671,19.854167,7.224307,19,59,4,0,0.816918,0.000157,0.000220
15701,2025-08-29 20:00:00,318.0700,318.0700,318.07,318.07,318519,2025-08-29,318.021555,318.026694,317.805193,...,0.064824,317.200799,48.980919,20,0,4,0,0.721395,-0.000094,0.000252


In [16]:
df.to_csv("30days_5min_complete_data.csv")

In [2]:
!curl -X GET "https://api.polygon.io/futures/vX/quotes/GCJ5?limit=10&sort=timestamp.desc&apiKey=AGSN41hnEcg8FzjPsHDAXaomluKbj5g0"

{"status":"NOT_AUTHORIZED","request_id":"f1dff183d059348090ea0cdb4641dda1","message":"You are not entitled to this data. Please upgrade your plan at https://polygon.io/pricing"}